<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distribution audit

Before defining any scoring rule, I inspect the distributions of the main page-level signals used in the capstone.

The purpose is to understand typical values, spread, skew, and possible heavy tails in variables such as impressions, click-through rate, average position, content age, days since last update, and word count.

These distribution checks are descriptive. They help identify whether simple thresholds could be unstable or overly sensitive to extreme values.

In [4]:
import os
import subprocess
import pandas as pd

REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

data_path = os.path.join(
    REPO_DIR,
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

signal_cols = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

distribution_table = df[signal_cols].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).T

print(distribution_table.round(3))

print("\nMissing values:")
print(df[signal_cols].isna().sum())

                          count      mean        std   min     25%      50%  \
impressions_90d         30000.0  5200.366  16838.020   1.0    81.0   731.00   
ctr                     30000.0     0.511      3.279   0.0     0.0     0.07   
avg_position            30000.0    16.342     15.217   0.0     6.2    10.80   
content_age_days        30000.0   256.168    132.708  90.0   132.0   236.00   
days_since_last_update  30000.0    46.098     42.079   1.0    20.0    20.00   
word_count              22301.0  3107.760   1452.383   8.0  2413.0  2877.00   

                            75%       90%       95%        99%       max  
impressions_90d         3615.25  12136.40  22996.50  73505.830  517715.0  
ctr                        0.29      0.65      1.09      8.330     100.0  
avg_position              22.30     36.80     48.20     69.901     245.0  
content_age_days         333.00    463.00    487.00    537.000     564.0  
days_since_last_update   104.00    104.00    104.00    106.000     373.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal tests

I test three public-safe signals against the observed `trend_direction` label.

**Signal 1 — Content staleness:** pages that have not been updated recently may be more likely to be declining.

**Signal 2 — CTR relative to average position:** weak click-through rate may indicate underperformance, but CTR should be interpreted together with search position because expected CTR changes by rank.

**Signal 3 — Search visibility:** pages with different levels of recent impressions may show different decline rates, but impressions alone should not be treated as a direct cause of decline.

Each signal is evaluated descriptively using the observed share of pages whose `trend_direction` is `"down"`. The verdicts are directional evidence only, not causal claims.

In [5]:
import numpy as np

df["actual_decline"] = (df["trend_direction"] == "down").astype(int)

# -----------------------------
# Signal 1: staleness
# -----------------------------
stale_bins = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=["0-30", "31-90", "91-180", "181-365", "365+"]
)

stale_table = (
    df.assign(staleness_bucket=stale_bins)
      .groupby("staleness_bucket", observed=False)
      .agg(
          pages=("content_id", "count"),
          declining_rate=("actual_decline", "mean")
      )
)

print("SIGNAL 1 — STALENESS")
print(stale_table.round(3))


# -----------------------------
# Signal 2: CTR relative to position
# -----------------------------
position_bins = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "50+"],
    include_lowest=True
)

ctr_table = (
    df.assign(position_bucket=position_bins)
      .groupby("position_bucket", observed=False)
      .agg(
          pages=("content_id", "count"),
          median_ctr=("ctr", "median"),
          declining_rate=("actual_decline", "mean")
      )
)

print("\nSIGNAL 2 — CTR RELATIVE TO POSITION")
print(ctr_table.round(3))


# -----------------------------
# Signal 3: impressions
# -----------------------------
impression_bins = pd.qcut(
    df["impressions_90d"],
    q=5,
    duplicates="drop"
)

impression_table = (
    df.assign(impression_bucket=impression_bins)
      .groupby("impression_bucket", observed=False)
      .agg(
          pages=("content_id", "count"),
          median_impressions=("impressions_90d", "median"),
          declining_rate=("actual_decline", "mean")
      )
)

print("\nSIGNAL 3 — IMPRESSIONS")
print(impression_table.round(3))

SIGNAL 1 — STALENESS
                  pages  declining_rate
staleness_bucket                       
0-30              20480           0.511
31-90               175           0.589
91-180             9171           0.611
181-365             169           0.467
365+                  5           0.600

SIGNAL 2 — CTR RELATIVE TO POSITION
                 pages  median_ctr  declining_rate
position_bucket                                   
1-3               2346        0.00           0.246
4-10             11842        0.16           0.569
11-20             7273        0.10           0.610
21-50             7225        0.03           0.562
50+               1314        0.00           0.343

SIGNAL 3 — IMPRESSIONS
                    pages  median_impressions  declining_rate
impression_bucket                                            
(0.999, 39.0]        6041                 5.0           0.325
(39.0, 364.0]        5964               150.0           0.603
(364.0, 1375.0]      5997        

### Signal verdicts

**Signal 1 — Content staleness: MIXED**

The observed decline rate increases from 51.1% for pages updated within 0–30 days to 61.1% for pages in the 91–180 day bucket, which supports the idea that stale content can be associated with decline. However, the pattern is not monotonic, and some older buckets contain very few pages. Staleness is therefore useful as a review signal, but not as a standalone rule.

**Signal 2 — CTR relative to position: MIXED**

Pages in positions 11–20 show the highest observed decline rate at about 61.0%, while pages in positions 1–3 have a much lower decline rate of about 24.6%. However, the relationship changes across position groups and is not monotonic. CTR and position together appear informative, but should be interpreted jointly rather than as a single threshold.

**Signal 3 — Impressions: MIXED**

Observed decline rates differ substantially across impression groups, rising from about 32.5% in the lowest-impression bucket to 63.3% in the fourth bucket before decreasing to 54.6% in the highest bucket. This suggests impressions contain useful directional information, but higher impressions do not consistently correspond to higher or lower decline risk.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test: stale content

The Week-4 baseline assigns a staleness signal to pages that have not been updated recently. I therefore test whether pages classified as stale actually show a higher observed decline rate than less-stale pages.

For this audit, I use 90 days as the operational threshold. This is a rule for decision support, not a causal claim that age or lack of updating causes performance decline.

In [6]:
stale_flag = df["days_since_last_update"] > 90

flag_test = (
    df.assign(stale_flag=stale_flag)
      .groupby("stale_flag")
      .agg(
          pages=("content_id", "count"),
          declining_rate=("actual_decline", "mean"),
          median_days_since_update=("days_since_last_update", "median")
      )
)

print("FLAG-LINKED TEST — STALENESS")
print(flag_test.round(3))

non_stale_rate = flag_test.loc[False, "declining_rate"]
stale_rate = flag_test.loc[True, "declining_rate"]

print("\nDecline-rate difference:",
      round(stale_rate - non_stale_rate, 3))

FLAG-LINKED TEST — STALENESS
            pages  declining_rate  median_days_since_update
stale_flag                                                 
False       20655           0.512                      20.0
True         9345           0.608                     104.0

Decline-rate difference: 0.096


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Practical interpretation

The staleness flag is directionally supported by the observed data: pages more than 90 days since their last update had a decline rate of about 60.8%, compared with 51.2% for less-stale pages.

This suggests that staleness can be useful for prioritizing pages for human review, but it should not be used alone. A content team should combine it with signals such as CTR, average position, impressions, and editorial context before deciding whether a page actually needs to be refreshed.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.